In [1]:
# ============================================================
# HW5: Fine-Tuning an LLM Using LoRA
# Personal Parameters and Reproducibility Setup
# ============================================================

# Personal parameters derived from the last four digits of the SJSU ID
SID4 = 1980
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print("Personal Parameters")
print("-------------------")
print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}")
print(f"CLS_A = {CLS_A}")
print(f"CLS_B = {CLS_B}")

Personal Parameters
-------------------
SID4  = 1980
SEED  = 1980
SLICE = 980
HP_ID = 0
CLS_A = 0
CLS_B = 1


In [2]:
# ============================================================
# Verify GPU Availability
# ============================================================

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU is not available. Please enable GPU in Runtime settings.")

Selected device: cuda
GPU name: Tesla T4
CUDA version: 12.8


In [3]:
# Install libraries required for this text-only LoRA assignment
!pip uninstall -y -q torchvision torchaudio torchao

!pip install -q -U \
    transformers \
    datasets \
    peft \
    accelerate \
    sentencepiece \
    evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [4]:
# ============================================================
# Step 2: Import Libraries and Set Random Seeds
# ============================================================

import os
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

# Set random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Libraries imported successfully.")
print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Libraries imported successfully.
Selected device: cuda
GPU: Tesla T4


# Part 1: Data Preparation

In this part, we load the DialogSum dataset and convert each example
into an input-target format.

- Input: dialogue
- Target: summary

In [5]:
# ============================================================
# Part 1: Data Preparation
# Part 1.1: Load the DialogSum Dataset
# ============================================================

# Load the DialogSum dataset specified in the assignment
dataset = load_dataset("neil-code/dialogsum-test")

# Display the dataset structure
print("Dataset loaded successfully.")
print(dataset)

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/441k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/499 [00:00<?, ? examples/s]

Dataset loaded successfully.
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})


In [6]:
# ============================================================
# Part 1.2: Inspect the Dataset
# ============================================================

# Identify the first available dataset split
split_name = list(dataset.keys())[0]
raw_data = dataset[split_name]

print("Selected split:", split_name)
print("Number of examples:", len(raw_data))
print("Dataset columns:", raw_data.column_names)

# Display the first raw example
print("\nFirst raw example:")
print(raw_data[0])

Selected split: train
Number of examples: 1999
Dataset columns: ['id', 'dialogue', 'summary', 'topic']

First raw example:
{'id': 'train_0', 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications 

In [7]:
# ============================================================
# Part 1.3: Convert Dataset into Input-Target Format
# ============================================================

def process_example(example):
    """
    Convert one raw DialogSum example into the format required
    for supervised summarization.

    input  = dialogue given to the model
    target = reference summary expected from the model
    """

    return {
        "input": example["dialogue"],
        "target": example["summary"]
    }


# Apply the conversion to every example
processed_data = raw_data.map(
    process_example,
    remove_columns=raw_data.column_names
)

print("Dataset converted successfully.")
print("Processed columns:", processed_data.column_names)

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Dataset converted successfully.
Processed columns: ['input', 'target']


In [8]:
# ============================================================
# Part 1.4: Display Two Processed Samples
# ============================================================

for i in range(2):
    print("\n" + "=" * 80)
    print(f"PROCESSED SAMPLE {i + 1}")
    print("=" * 80)

    print("\nINPUT DIALOGUE:")
    print(processed_data[i]["input"])

    print("\nTARGET SUMMARY:")
    print(processed_data[i]["target"])


PROCESSED SAMPLE 1

INPUT DIALOGUE:
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

TARGET SUMMARY:
Mr. Smith's ge

### Part 1 Interpretation

The DialogSum dataset was successfully loaded and converted into
input-target pairs.

The dialogue is used as the model input, while the human-written
summary is used as the target output. Displaying two examples confirms
that the data has the correct format for supervised summarization.

This processed dataset will be used in the later fine-tuning stage.

# Part 2: Baseline Inference

A pretrained FLAN-T5 model is used to generate summaries before
fine-tuning. These results will be compared with the summaries
generated after LoRA fine-tuning.

In [9]:
# ============================================================
# Part 2: Baseline Inference
# Part 2.1: Load the Pretrained Model
# ============================================================

# Select a small pretrained sequence-to-sequence language model
MODEL_NAME = "google/flan-t5-small"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the pretrained model
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Move the model to the GPU if available
baseline_model = baseline_model.to(device)

# Set the model to evaluation mode
baseline_model.eval()

print("Pretrained model loaded successfully.")
print("Model name:", MODEL_NAME)
print("Running device:", device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Pretrained model loaded successfully.
Model name: google/flan-t5-small
Running device: cuda


In [10]:
# ============================================================
# Part 2.2: Select Two Dialogues for Inference
# ============================================================

# Select the first two examples from the processed dataset
baseline_examples = [
    processed_data[0],
    processed_data[1]
]

print("Two dialogues selected for baseline inference.")

for i, example in enumerate(baseline_examples, start=1):
    print("\n" + "=" * 80)
    print(f"DIALOGUE {i}")
    print("=" * 80)
    print(example["input"])

    print("\nREFERENCE SUMMARY:")
    print(example["target"])

Two dialogues selected for baseline inference.

DIALOGUE 1
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

REFERENC

In [11]:
# ============================================================
# Part 2.3: Define the Summary Generation Function
# ============================================================

def generate_summary(
    model,
    dialogue,
    max_input_length=512,
    max_output_length=128
):
    """
    Generate a summary for one dialogue.

    Parameters:
        model: pretrained or fine-tuned sequence-to-sequence model
        dialogue: input conversation
        max_input_length: maximum number of input tokens
        max_output_length: maximum number of output tokens

    Returns:
        Generated summary as text
    """

    # Add an instruction so FLAN-T5 understands the task
    prompt = "Summarize the following dialogue:\n" + dialogue

    # Tokenize the input dialogue
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    )

    # Move the tokenized input to the same device as the model
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate the summary without calculating gradients
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=max_output_length,
            num_beams=4,
            early_stopping=True
        )

    # Convert generated token IDs into readable text
    generated_summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return generated_summary

In [12]:
# ============================================================
# Part 2.4: Generate Baseline Summaries
# ============================================================

# Store the summaries generated before fine-tuning
baseline_outputs = []

for example in baseline_examples:
    summary = generate_summary(
        model=baseline_model,
        dialogue=example["input"]
    )

    baseline_outputs.append(summary)

print("Baseline summaries generated successfully.")

Baseline summaries generated successfully.


In [13]:
# ============================================================
# Part 2.5: Display Baseline Results
# ============================================================

for i, (example, generated_summary) in enumerate(
    zip(baseline_examples, baseline_outputs),
    start=1
):
    print("\n" + "=" * 80)
    print(f"BASELINE RESULT {i}")
    print("=" * 80)

    print("\nINPUT DIALOGUE:")
    print(example["input"])

    print("\nREFERENCE SUMMARY:")
    print(example["target"])

    print("\nGENERATED BASELINE SUMMARY:")
    print(generated_summary)


BASELINE RESULT 1

INPUT DIALOGUE:
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

REFERENCE SUMMARY:
Mr. Smith's 

### Part 2 Interpretation

The pretrained FLAN-T5 model generated summaries for the two selected
dialogues before any fine-tuning.

These summaries are the baseline results. Because the model has not been
trained specifically on the DialogSum dataset, its outputs may omit
important details, contain general statements, or differ from the
reference summaries.

The same two dialogues and the same baseline outputs will be used later
to compare the effects of LoRA fine-tuning.

# Part 3: Apply LoRA

LoRA is a parameter-efficient fine-tuning method. Instead of updating
all parameters of the pretrained model, LoRA adds small trainable
adapter matrices to selected layers.

For the first experiment, we use:

- LoRA rank: r = 4
- LoRA alpha: 16
- LoRA dropout: 0.1
- Target modules: query and value projection layers

In [14]:
# ============================================================
# Part 3: Apply LoRA
# Part 3.1: Load a Separate Model for LoRA
# ============================================================

# Load a fresh copy of the pretrained model.
# This keeps the baseline model unchanged.
lora_base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Move the model to the selected device
lora_base_model = lora_base_model.to(device)

print("Separate pretrained model loaded for LoRA.")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Separate pretrained model loaded for LoRA.


In [15]:
# ============================================================
# Part 3.2: Define the LoRA Configuration
# ============================================================

# LoRA configuration for the first experiment
lora_config_r4 = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,

    # Rank of the LoRA update matrices
    r=4,

    # Scaling factor for the LoRA updates
    lora_alpha=16,

    # Dropout applied to LoRA layers during training
    lora_dropout=0.1,

    # Apply LoRA to query and value projection layers
    target_modules=["q", "v"],

    # Do not train the original bias parameters
    bias="none"
)

print("LoRA configuration created.")
print(lora_config_r4)

LoRA configuration created.
LoraConfig(task_type=<TaskType.SEQ_2_SEQ_LM: 'SEQ_2_SEQ_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=4, target_modules={'v', 'q'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


In [16]:
# ============================================================
# Part 3.3: Attach the LoRA Adapter
# ============================================================

# Attach the LoRA adapter to the pretrained model
lora_model_r4 = get_peft_model(
    lora_base_model,
    lora_config_r4
)

# Move the LoRA model to the selected device
lora_model_r4 = lora_model_r4.to(device)

print("LoRA adapter attached successfully.")

LoRA adapter attached successfully.


In [17]:
# ============================================================
# Part 3.4: Display Total and Trainable Parameters
# ============================================================

# Display the number of trainable and total parameters
lora_model_r4.print_trainable_parameters()

# Calculate parameter counts manually for reporting
total_parameters = sum(
    parameter.numel()
    for parameter in lora_model_r4.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in lora_model_r4.parameters()
    if parameter.requires_grad
)

trainable_percentage = (
    100 * trainable_parameters / total_parameters
)

print("\nParameter Summary")
print("-----------------")
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Trainable percentage: {trainable_percentage:.4f}%")

trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230

Parameter Summary
-----------------
Total parameters: 77,133,184
Trainable parameters: 172,032
Trainable percentage: 0.2230%


### Part 3 Interpretation

The LoRA adapter was successfully attached to the pretrained FLAN-T5
model. The original model parameters are frozen, and only the LoRA
parameters are trainable.

Therefore, the number of trainable parameters should be much smaller
than the total number of parameters. This reduces memory usage and
training time while allowing the model to adapt to the DialogSum
summarization task.

The model with rank r = 4 will be fine-tuned in Part 4. A second model
with rank r = 16 will be trained later in Part 7 for comparison.

# Part 4: Fine-Tuning

The LoRA-adapted FLAN-T5 model is fine-tuned on a small subset of the
DialogSum dataset. A subset is used because this assignment focuses on
demonstrating LoRA fine-tuning efficiently.

In [18]:
# ============================================================
# Part 4: Fine-Tuning
# Part 4.1: Prepare the Training Subset
# ============================================================

# Use a reproducibly shuffled subset for training
shuffled_data = processed_data.shuffle(seed=SEED)

# Use a small subset to reduce training time in Google Colab
TRAIN_SIZE = min(300, len(shuffled_data))

train_data = shuffled_data.select(range(TRAIN_SIZE))

print("Training examples:", len(train_data))
print("Training seed:", SEED)

Training examples: 300
Training seed: 1980


The dataset is shuffled using SEED = 1980, so the same training examples are selected whenever the notebook is rerun under the same conditions.

In [19]:
# ============================================================
# Part 4.2: Tokenize the Training Dataset
# ============================================================

def tokenize_function(batch):
    """
    Tokenize dialogues as model inputs and summaries as labels.
    """

    # Add a summarization instruction to each dialogue
    prompts = [
        "Summarize the following dialogue:\n" + dialogue
        for dialogue in batch["input"]
    ]

    # Tokenize the input dialogues
    model_inputs = tokenizer(
        prompts,
        max_length=512,
        truncation=True
    )

    # Tokenize the target summaries
    labels = tokenizer(
        text_target=batch["target"],
        max_length=128,
        truncation=True
    )

    # Store tokenized summaries as training labels
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


# Apply tokenization to the training dataset
tokenized_train_data = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=train_data.column_names
)

print("Training data tokenized successfully.")
print(tokenized_train_data)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Training data tokenized successfully.
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 300
})


In [20]:
# ============================================================
# Part 4.3: Create the Data Collator
# ============================================================

# The data collator dynamically pads inputs and labels within each batch
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model_r4
)

print("Data collator created successfully.")

Data collator created successfully.


In [21]:
# ============================================================
# Part 4.4: Define Training Arguments
# ============================================================

training_args = TrainingArguments(
    # Directory used for temporary training files
    output_dir="./dialogsum_lora_r4_output",

    # Train examples in small batches to fit GPU memory
    per_device_train_batch_size=4,

    # Accumulate gradients over two batches
    gradient_accumulation_steps=2,

    # Assignment uses a small training budget
    num_train_epochs=1,

    # Learning rate for LoRA adapter training
    learning_rate=1e-4,

    # Log training loss periodically
    logging_steps=25,

    # Do not save intermediate checkpoints
    save_strategy="no",

    # Disable external experiment tracking
    report_to="none",

    # Use mixed precision when a CUDA GPU is available
    # Disable mixed precision because it caused NaN gradients
    fp16=False,
    bf16=False,

    # Make Trainer operations reproducible
    seed=SEED,

    # Keep unused dataset columns from causing issues
    remove_unused_columns=True
)

print("Training arguments created successfully.")

Training arguments created successfully.


In [22]:
# ============================================================
# Part 4.5: Create the Trainer
# ============================================================

# Disable the cache during training
lora_model_r4.config.use_cache = False

trainer_r4 = Trainer(
    model=lora_model_r4,
    args=training_args,
    train_dataset=tokenized_train_data,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Trainer created successfully.")

Trainer created successfully.


In [23]:
# ============================================================
# Part 4.6: Fine-Tune the Model
# ============================================================

print("Starting LoRA fine-tuning with rank r=4...")

train_result_r4 = trainer_r4.train()

print("Fine-tuning completed.")

Starting LoRA fine-tuning with rank r=4...


Step,Training Loss
25,4.868895


Fine-tuning completed.


In [24]:
# ============================================================
# Part 4.7: Display Training Results
# ============================================================

# Extract the training metrics
training_metrics_r4 = train_result_r4.metrics

print("Training Metrics")
print("----------------")
for metric_name, metric_value in training_metrics_r4.items():
    print(f"{metric_name}: {metric_value}")

# Display the final training loss separately
if "train_loss" in training_metrics_r4:
    print("\nFinal Training Loss:", training_metrics_r4["train_loss"])

Training Metrics
----------------
train_runtime: 22.1663
train_samples_per_second: 13.534
train_steps_per_second: 1.714
total_flos: 36838139510784.0
train_loss: 4.768120815879421
epoch: 1.0

Final Training Loss: 4.768120815879421


In [25]:
# Display the recorded training logs
print("\nTraining Log History")
print("--------------------")

for log_entry in trainer_r4.state.log_history:
    print(log_entry)


Training Log History
--------------------
{'loss': 4.868894958496094, 'grad_norm': 1.556963324546814, 'learning_rate': 3.6842105263157895e-05, 'epoch': 0.6666666666666666, 'step': 25}
{'train_runtime': 22.1663, 'train_samples_per_second': 13.534, 'train_steps_per_second': 1.714, 'total_flos': 36838139510784.0, 'train_loss': 4.768120815879421, 'epoch': 1.0, 'step': 38}


In [26]:
# ============================================================
# Part 4.8: Save the Fine-Tuned LoRA Adapter
# ============================================================

# Save only the trained LoRA adapter
R4_ADAPTER_PATH = "./dialogsum_lora_r4_adapter"

lora_model_r4.save_pretrained(R4_ADAPTER_PATH)
tokenizer.save_pretrained(R4_ADAPTER_PATH)

print("Fine-tuned LoRA adapter saved.")
print("Saved location:", R4_ADAPTER_PATH)

Fine-tuned LoRA adapter saved.
Saved location: ./dialogsum_lora_r4_adapter


### Part 4 Interpretation

The LoRA-adapted FLAN-T5 model was fine-tuned on a DialogSum subset.
Only the LoRA parameters were updated while the original pretrained
model parameters remained frozen.

The final training loss shows how well the model learned the training
examples during the one-epoch training run. The adapter was saved
separately so that it can be loaded later without saving a complete copy
of the original model.

This trained rank-4 model will be evaluated in Part 5.

# Part 5: Post-Fine-Tuning Inference

The fine-tuned LoRA model is used to generate summaries for the same
two dialogues used during baseline inference.

In [27]:
# ============================================================
# Part 5: Post-Fine-Tuning Inference
# Part 5.1: Prepare the Fine-Tuned Model
# ============================================================

# Enable the model cache for faster generation
lora_model_r4.config.use_cache = True

# Set the model to evaluation mode
lora_model_r4.eval()

print("Fine-tuned LoRA model is ready for inference.")

Fine-tuned LoRA model is ready for inference.


In [28]:
# ============================================================
# Part 5.2: Generate Fine-Tuned Summaries
# ============================================================

# Store summaries generated after fine-tuning
finetuned_outputs_r4 = []

for example in baseline_examples:
    summary = generate_summary(
        model=lora_model_r4,
        dialogue=example["input"]
    )

    finetuned_outputs_r4.append(summary)

print("Fine-tuned summaries generated successfully.")

Fine-tuned summaries generated successfully.


In [29]:
# ============================================================
# Part 5.3: Display Post-Fine-Tuning Results
# ============================================================

for i, (example, generated_summary) in enumerate(
    zip(baseline_examples, finetuned_outputs_r4),
    start=1
):
    print("\n" + "=" * 80)
    print(f"POST-FINE-TUNING RESULT {i}")
    print("=" * 80)

    print("\nINPUT DIALOGUE:")
    print(example["input"])

    print("\nREFERENCE SUMMARY:")
    print(example["target"])

    print("\nGENERATED SUMMARY AFTER LoRA FINE-TUNING:")
    print(generated_summary)


POST-FINE-TUNING RESULT 1

INPUT DIALOGUE:
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

REFERENCE SUMMARY:
Mr. 

### Part 5 Interpretation

The fine-tuned rank-4 LoRA model generated summaries for the same two
dialogues used during baseline inference.

For these two examples, the generated summaries after fine-tuning were
the same as the baseline summaries. Therefore, no visible improvement
was observed in these examples. This may be because the model was trained
for only one epoch using a relatively small subset of the dataset.

# Part 6: Output Comparison

The baseline summaries from Part 2 are compared with the summaries
generated by the fine-tuned LoRA model from Part 5.

In [30]:
# ============================================================
# Part 6: Output Comparison
# Part 6.1: Create a Comparison Table
# ============================================================

import pandas as pd

comparison_table = pd.DataFrame({
    "Dialogue": ["Dialogue 1", "Dialogue 2"],
    "Reference Summary": [
        example["target"]
        for example in baseline_examples
    ],
    "Before Fine-Tuning": baseline_outputs,
    "After LoRA Fine-Tuning (r=4)": finetuned_outputs_r4
})

# Display the comparison table
comparison_table

,Dialogue,Reference Summary,Before Fine-Tuning,After LoRA Fine-Tuning (r=4)
0,Dialogue 1,"Mr. Smith's getting a check-up, and Doctor Haw...",Get a check-up.,Get a check-up.
1,Dialogue 2,Mrs Parker takes Ricky for his vaccines. Dr. P...,Ricky and I are going to see a doctor.,Ricky and I are going to see a doctor.


In [31]:
# ============================================================
# Part 6.2: Display Before-and-After Comparisons
# ============================================================

for i, example in enumerate(baseline_examples):
    print("\n" + "=" * 90)
    print(f"COMPARISON FOR DIALOGUE {i + 1}")
    print("=" * 90)

    print("\nREFERENCE SUMMARY:")
    print(example["target"])

    print("\nBEFORE LoRA FINE-TUNING:")
    print(baseline_outputs[i])

    print("\nAFTER LoRA FINE-TUNING:")
    print(finetuned_outputs_r4[i])


COMPARISON FOR DIALOGUE 1

REFERENCE SUMMARY:
Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.

BEFORE LoRA FINE-TUNING:
Get a check-up.

AFTER LoRA FINE-TUNING:
Get a check-up.

COMPARISON FOR DIALOGUE 2

REFERENCE SUMMARY:
Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.

BEFORE LoRA FINE-TUNING:
Ricky and I are going to see a doctor.

AFTER LoRA FINE-TUNING:
Ricky and I are going to see a doctor.


In [32]:
# ============================================================
# Part 6.3: Save the Comparison Results
# ============================================================

comparison_table.to_csv(
    "dialogsum_baseline_vs_lora_r4.csv",
    index=False
)

print("Comparison table saved as:")
print("dialogsum_baseline_vs_lora_r4.csv")

Comparison table saved as:
dialogsum_baseline_vs_lora_r4.csv


### Part 6 Interpretation

Before fine-tuning, the model generated short and incomplete summaries.
For example, the first summary only mentioned getting a check-up and did
not mention the yearly recommendation or smoking-cessation information.

After LoRA fine-tuning, the outputs for both dialogues remained the same
as the baseline outputs. Therefore, no visible improvement was observed
for these two examples.

The unchanged outputs may be due to the small training subset, one
training epoch, and the limited number of examples used for evaluation.

# Part 7: LoRA Rank Experiment

The rank-4 and rank-16 LoRA models are compared to study how LoRA
capacity affects the generated summary quality.

In [33]:
# ============================================================
# Part 7: LoRA Rank Experiment
# Part 7.1: Free Unnecessary GPU Memory
# ============================================================

# Remove objects that are no longer needed for training
del trainer_r4
del lora_base_model
del baseline_model

# Release unused GPU memory
torch.cuda.empty_cache()

print("Unused GPU memory released.")

Unused GPU memory released.


In [34]:
# ============================================================
# Part 7.2: Create the Rank-16 LoRA Model
# ============================================================

# Load a fresh copy of the pretrained model
lora_base_model_r16 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
lora_base_model_r16 = lora_base_model_r16.to(device)

# Define the rank-16 LoRA configuration.
# All settings are kept the same as r=4 except the rank.
lora_config_r16 = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,

    # Larger LoRA rank
    r=16,

    # Keep alpha the same for a fair comparison
    lora_alpha=16,

    # Keep dropout the same
    lora_dropout=0.1,

    # Use the same target modules
    target_modules=["q", "v"],

    bias="none"
)

# Attach the rank-16 LoRA adapter
lora_model_r16 = get_peft_model(
    lora_base_model_r16,
    lora_config_r16
)

lora_model_r16 = lora_model_r16.to(device)

print("Rank-16 LoRA model created successfully.")
lora_model_r16.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Rank-16 LoRA model created successfully.
trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


In [35]:
# ============================================================
# Part 7.3: Fine-Tune the Rank-16 Model
# ============================================================

# Disable the cache during training
lora_model_r16.config.use_cache = False

# Use the same training settings as the rank-4 model
training_args_r16 = TrainingArguments(
    output_dir="./dialogsum_lora_r16_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=1e-4,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
    fp16=False,
    bf16=False,
    seed=SEED,
    remove_unused_columns=True
)

# Create the Trainer for rank 16
trainer_r16 = Trainer(
    model=lora_model_r16,
    args=training_args_r16,
    train_dataset=tokenized_train_data,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Starting LoRA fine-tuning with rank r=16...")

train_result_r16 = trainer_r16.train()

print("Rank-16 fine-tuning completed.")

Starting LoRA fine-tuning with rank r=16...


Step,Training Loss
25,4.869791


Rank-16 fine-tuning completed.


In [36]:
# ============================================================
# Part 7.4: Display Rank-16 Training Results
# ============================================================

training_metrics_r16 = train_result_r16.metrics

print("Rank-16 Training Metrics")
print("------------------------")

for metric_name, metric_value in training_metrics_r16.items():
    print(f"{metric_name}: {metric_value}")

if "train_loss" in training_metrics_r16:
    print("\nRank-16 Final Training Loss:",
          training_metrics_r16["train_loss"])

Rank-16 Training Metrics
------------------------
train_runtime: 22.48
train_samples_per_second: 13.345
train_steps_per_second: 1.69
total_flos: 37151438684160.0
train_loss: 4.767175774825247
epoch: 1.0

Rank-16 Final Training Loss: 4.767175774825247


In [37]:
# ============================================================
# Part 7.5: Save the Rank-16 LoRA Adapter
# ============================================================

R16_ADAPTER_PATH = "./dialogsum_lora_r16_adapter"

lora_model_r16.save_pretrained(R16_ADAPTER_PATH)
tokenizer.save_pretrained(R16_ADAPTER_PATH)

print("Rank-16 LoRA adapter saved.")
print("Saved location:", R16_ADAPTER_PATH)

Rank-16 LoRA adapter saved.
Saved location: ./dialogsum_lora_r16_adapter


In [38]:
# ============================================================
# Part 7.6: Generate Rank-16 Summaries
# ============================================================

# Enable the cache for faster generation
lora_model_r16.config.use_cache = True
lora_model_r16.eval()

# Generate summaries for the same two dialogues
finetuned_outputs_r16 = []

for example in baseline_examples:
    summary = generate_summary(
        model=lora_model_r16,
        dialogue=example["input"]
    )

    finetuned_outputs_r16.append(summary)

print("Rank-16 summaries generated successfully.")

Rank-16 summaries generated successfully.


In [39]:
# ============================================================
# Part 7.7: Compare Rank-4 and Rank-16 Outputs
# ============================================================

rank_comparison_table = pd.DataFrame({
    "Dialogue": ["Dialogue 1", "Dialogue 2"],
    "Reference Summary": [
        example["target"]
        for example in baseline_examples
    ],
    "LoRA r=4": finetuned_outputs_r4,
    "LoRA r=16": finetuned_outputs_r16
})

rank_comparison_table

,Dialogue,Reference Summary,LoRA r=4,LoRA r=16
0,Dialogue 1,"Mr. Smith's getting a check-up, and Doctor Haw...",Get a check-up.,Get a check-up.
1,Dialogue 2,Mrs Parker takes Ricky for his vaccines. Dr. P...,Ricky and I are going to see a doctor.,Ricky and I are going to see a doctor.


In [40]:
# ============================================================
# Part 7.8: Display Rank Comparison Clearly
# ============================================================

for i, example in enumerate(baseline_examples):
    print("\n" + "=" * 90)
    print(f"RANK COMPARISON FOR DIALOGUE {i + 1}")
    print("=" * 90)

    print("\nREFERENCE SUMMARY:")
    print(example["target"])

    print("\nLoRA r=4 SUMMARY:")
    print(finetuned_outputs_r4[i])

    print("\nLoRA r=16 SUMMARY:")
    print(finetuned_outputs_r16[i])


RANK COMPARISON FOR DIALOGUE 1

REFERENCE SUMMARY:
Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.

LoRA r=4 SUMMARY:
Get a check-up.

LoRA r=16 SUMMARY:
Get a check-up.

RANK COMPARISON FOR DIALOGUE 2

REFERENCE SUMMARY:
Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.

LoRA r=4 SUMMARY:
Ricky and I are going to see a doctor.

LoRA r=16 SUMMARY:
Ricky and I are going to see a doctor.


In [41]:
# ============================================================
# Part 7.9: Save Rank Comparison Results
# ============================================================

rank_comparison_table.to_csv(
    "dialogsum_lora_rank_comparison.csv",
    index=False
)

print("Rank comparison saved as:")
print("dialogsum_lora_rank_comparison.csv")

Rank comparison saved as:
dialogsum_lora_rank_comparison.csv


### Part 7 Interpretation

The rank-4 and rank-16 models used the same dataset, learning rate,
dropout, number of epochs, and target modules. The only major difference
was the LoRA rank.

The rank-4 model trained 172,032 parameters, while the rank-16 model
trained 688,128 parameters. The rank-16 model therefore trained four
times more adapter parameters. Its training loss was slightly lower:
4.7672 compared with 4.7681 for rank 4.

However, both ranks produced the same summaries for the two evaluation
dialogues. Therefore, increasing the rank from 4 to 16 did not produce a
visible quality improvement in this experiment.

## Overall Conclusion

This assignment demonstrated parameter-efficient fine-tuning of the
pretrained FLAN-T5 model using LoRA on the DialogSum dataset.

The model was evaluated before and after fine-tuning, and two LoRA ranks
were compared. The rank-4 model trained 172,032 parameters, while the
rank-16 model trained 688,128 parameters. Both models used far fewer
trainable parameters than the full model.

Although both LoRA models achieved valid training losses, their outputs
for the two selected dialogues did not visibly change from the baseline.
This suggests that one epoch and a small training subset were not enough
to produce an observable improvement on these examples.

The experiment still demonstrates that LoRA can fine-tune a language
model while updating only a small fraction of the original parameters.

In [42]:
# Compress both LoRA adapter folders into one ZIP file
!zip -r HW5_LoRA_Adapters.zip \
    dialogsum_lora_r4_adapter \
    dialogsum_lora_r16_adapter

  adding: dialogsum_lora_r4_adapter/ (stored 0%)
  adding: dialogsum_lora_r4_adapter/adapter_config.json (deflated 60%)
  adding: dialogsum_lora_r4_adapter/README.md (deflated 66%)
  adding: dialogsum_lora_r4_adapter/adapter_model.safetensors (deflated 8%)
  adding: dialogsum_lora_r4_adapter/tokenizer_config.json (deflated 83%)
  adding: dialogsum_lora_r4_adapter/tokenizer.json (deflated 75%)
  adding: dialogsum_lora_r16_adapter/ (stored 0%)
  adding: dialogsum_lora_r16_adapter/adapter_config.json (deflated 60%)
  adding: dialogsum_lora_r16_adapter/README.md (deflated 66%)
  adding: dialogsum_lora_r16_adapter/adapter_model.safetensors (deflated 7%)
  adding: dialogsum_lora_r16_adapter/tokenizer_config.json (deflated 83%)
  adding: dialogsum_lora_r16_adapter/tokenizer.json (deflated 75%)


In [43]:
from google.colab import files

files.download("HW5_LoRA_Adapters.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>